In [1]:

from sklearn.preprocessing import LabelEncoder
import seaborn as sns

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

solar_d = pd.read_csv(
    '../data/fact_solar_daily.csv',
    parse_dates=['date']
)

solar_h = pd.read_csv(
    '../data/fact_solar_hourly.csv',
    parse_dates=['date', 'hour_ts']
)

weather_d = pd.read_csv(
    '../data/fact_weather_daily.csv',
    parse_dates=['date']
)

weather_h = pd.read_csv(
    '../data/fact_weather_hourly.csv',
    parse_dates=['date', 'hour_ts']
)

dim_date = pd.read_csv(
    '../data/dim_date.csv',
    parse_dates=['date']
)

wc_codes = pd.read_csv(
    '../data/dim_weather_codes.csv'
)

# Merge daily solar + weather
daily = pd.merge(
    solar_d,
    weather_d,
    on='date',
    how='inner'
)

# Add date dimension columns
daily = pd.merge(
    daily,
    dim_date[['date', 'month_name', 'season', 'is_weekend', 'day_name']],
    on='date',
    how='left'
)

print(daily.shape)


(89, 20)


In [5]:
df = daily.copy()

# Encode season (Dry=0, Wet=1)
df['season_enc']      = (df['season'] == 'Wet').astype(int)
df['is_weekend_enc'] = df['is_weekend'].fillna(False).astype(int)

# Sunshine efficiency ratio
df['sunshine_ratio'] = df['sunshine_duration'] / (df['daylight_duration'] + 1e-5)

# Radiation per cloud (interaction feature)
df['rad_clear'] = df['shortwave_radiation_sum'] * (1 - df['cloud_cover_mean']/100)

FEATURES = [
    'shortwave_radiation_sum', 'sunshine_duration',
    'cloud_cover_mean', 'temperature_2m_mean',
    'wind_speed_10m_mean', 'rain_sum',
    'season_enc', 'is_weekend_enc',
    'sunshine_ratio', 'rad_clear'
]
TARGET = 'generation_kwh'

X = df[FEATURES]
y = df[TARGET]


/var/folders/lh/gvpbf1b15yd43l_x_kk3zxgc0000gn/T/ipykernel_31993/425362064.py:5: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['is_weekend_enc'] = df['is_weekend'].fillna(False).astype(int)


In [4]:
df['is_weekend_enc'] = df['is_weekend'].astype(int)

ValueError: cannot convert float NaN to integer